In [3]:
"""
系统工作流程图生成代码
使用graphviz生成ESP32温度监控系统的完整流程图
"""

import graphviz
from datetime import datetime

def create_system_flowchart():
    """创建ESP32温度监控系统的工作流程图"""

    # 创建有向图
    dot = graphviz.Digraph(
        name='ESP32_Temperature_Control_System',
        format='png',
        graph_attr={
            'rankdir': 'TB',  # 从上到下布局
            'bgcolor': 'transparent',
            'fontname': 'Arial',
            'fontsize': '12',
            'label': f'ESP32 Temperature Monitoring & Control System\n({datetime.now().strftime("%Y-%m-%d")})',
            'labelloc': 't',
            'labeljust': 'c',
            'fontcolor': '#2C3E50',
        },
        node_attr={
            'shape': 'box',
            'style': 'rounded,filled',
            'fontname': 'Arial',
            'fontsize': '10',
            'fillcolor': 'lightblue',
            'color': '#2C3E50',
        },
        edge_attr={
            'fontname': 'Arial',
            'fontsize': '9',
            'color': '#34495E',
        }
    )

    # 添加主要节点
    dot.node('START', '系统启动',
             shape='circle',
             fillcolor='lightgreen',
             style='filled,bold')

    dot.node('INIT', '硬件初始化\n- WiFi配置\n- I2C初始化\n- GPIO设置\n- 串口配置',
             shape='box3d',
             fillcolor='#AED6F1')

    dot.node('WIFI_CONNECT', 'WiFi连接\nSSID: ard-citcea',
             shape='parallelogram',
             fillcolor='#85C1E9')

    dot.node('TELEGRAM_INIT', 'Telegram Bot初始化\n- Token验证\n- 安全证书配置',
             shape='parallelogram',
             fillcolor='#85C1E9')

    dot.node('MAIN_LOOP', '主循环启动\n(1ms周期)',
             shape='diamond',
             fillcolor='#F9E79F')

    # 电流测量链路
    with dot.subgraph(name='cluster_current') as c:
        c.attr(label='电流测量链路 (20ms周期)',
               style='filled',
               fillcolor='#E8F8F5',
               color='#1ABC9C')

        c.node('CURRENT_ADC', 'ADS1115 ADC采样\n- I2C读取\n- 860 SPS速率',
               shape='box')
        c.node('DC_REMOVE', '去除直流偏置\n减去1.65V基准',
               shape='ellipse')
        c.node('CURRENT_CALC', '电流计算\nI_inst = V × 30 A/V',
               shape='box')
        c.node('RMS_ACCUM', 'RMS累加\nquadratic_sum_rms += I_inst² × Δt',
               shape='box')
        c.node('RMS_CALC', 'RMS计算 (每5秒)\nIrms = √(50 × sum)',
               shape='box')
        c.node('NOISE_FILTER', '噪声过滤\nIrms < 0.1A → 清零',
               shape='diamond',
               fillcolor='#FAD7A0')
        c.node('MOVING_AVG', '滑动平均滤波\n250个周期平均',
               shape='box')
        c.node('CURRENT_OUTPUT', '输出滤波电流\nI_rms_filt',
               shape='box',
               fillcolor='#A9DFBF')

        # 电流链路连接
        c.edges([
            ('CURRENT_ADC', 'DC_REMOVE'),
            ('DC_REMOVE', 'CURRENT_CALC'),
            ('CURRENT_CALC', 'RMS_ACCUM'),
            ('RMS_ACCUM', 'RMS_CALC'),
            ('RMS_CALC', 'NOISE_FILTER'),
            ('NOISE_FILTER', 'MOVING_AVG'),
            ('MOVING_AVG', 'CURRENT_OUTPUT')
        ])

    # 温度测量链路
    with dot.subgraph(name='cluster_temp') as t:
        t.attr(label='温度测量链路 (5秒周期)',
               style='filled',
               fillcolor='#FDEDEC',
               color='#E74C3C')

        t.node('TEMP_ADC', 'LM35 V4 ADC采样\n- ESP32 ADC3(IO35)\n- 12位分辨率',
               shape='box')
        t.node('VOLT_CONV', '电压转换\nV = (raw/4095) × 3.3V',
               shape='ellipse')
        t.node('TEMP_CALC', '温度计算\nTempC = V × 100°C/V\n+1°C校准可选',
               shape='box')
        t.node('TEMP_OUTPUT', '输出温度值',
               shape='box',
               fillcolor='#F5B7B1')

        # 温度链路连接
        t.edges([
            ('TEMP_ADC', 'VOLT_CONV'),
            ('VOLT_CONV', 'TEMP_CALC'),
            ('TEMP_CALC', 'TEMP_OUTPUT')
        ])

    # 控制逻辑部分
    with dot.subgraph(name='cluster_control') as ctrl:
        ctrl.attr(label='温度控制逻辑',
                  style='filled',
                  fillcolor='#FEF9E7',
                  color='#F39C12')

        ctrl.node('CONTROL_CHECK', '控制模式检查\nAUTO/MANUAL',
                  shape='diamond',
                  fillcolor='#FAD7A0')
        ctrl.node('TEMP_COMPARE', '温度比较\nTmin=20°C, Tmax=25°C',
                  shape='diamond')
        ctrl.node('RELAY_ON', '继电器闭合\n加热器通电\nLED ON',
                  shape='box',
                  fillcolor='#82E0AA')
        ctrl.node('RELAY_OFF', '继电器断开\n加热器断电\nLED OFF',
                  shape='box',
                  fillcolor='#F1948A')
        ctrl.node('ALARM_CHECK', '报警检查\nTalarm=35°C',
                  shape='diamond',
                  fillcolor='#F5B7B1')
        ctrl.node('SEND_ALARM', '发送Telegram警报\n设置alarmSent标志',
                  shape='box',
                  fillcolor='#F1948A')
        ctrl.node('ALARM_RESET', '报警复位\n(Temp < Talarm-2°C)',
                  shape='box')

        # 控制逻辑连接
        ctrl.edges([
            ('CONTROL_CHECK', 'TEMP_COMPARE'),
            ('TEMP_COMPARE', 'RELAY_ON'),
            ('TEMP_COMPARE', 'RELAY_OFF'),
            ('TEMP_COMPARE', 'ALARM_CHECK'),
            ('ALARM_CHECK', 'SEND_ALARM'),
            ('ALARM_CHECK', 'ALARM_RESET')
        ])

    # 通信与输出部分
    with dot.subgraph(name='cluster_comm') as comm:
        comm.attr(label='通信与数据输出',
                  style='filled',
                  fillcolor='#EAF2F8',
                  color='#3498DB')

        comm.node('SERIAL_OUT', '串口数据输出\n格式: "Temp: XX.XX °C | Irms: X.XXXXX"',
                  shape='box')
        comm.node('TELEGRAM_POLL', 'Telegram消息轮询\n1秒间隔',
                  shape='parallelogram')
        comm.node('HANDLE_CMD', '处理Telegram命令\n/start, /relay_on, /relay_off, /state, /temp',
                  shape='box')
        comm.node('GUI_RECEIVE', 'Python GUI接收数据\nStreamlit可视化',
                  shape='box')
        comm.node('PRICE_API', '电价API查询\nREE西班牙电价',
                  shape='box')
        comm.node('COST_CALC', '用电成本计算\n基于实时电价',
                  shape='box')
        comm.node('USER_INTERFACE', '用户界面\n- 实时图表\n- 数据表格\n- 控制面板',
                  shape='box3d',
                  fillcolor='#D6EAF8')

        # 通信链路连接
        comm.edges([
            ('SERIAL_OUT', 'GUI_RECEIVE'),
            ('GUI_RECEIVE', 'PRICE_API'),
            ('PRICE_API', 'COST_CALC'),
            ('COST_CALC', 'USER_INTERFACE')
        ])

    # 主要流程连接
    dot.edges([
        ('START', 'INIT'),
        ('INIT', 'WIFI_CONNECT'),
        ('WIFI_CONNECT', 'TELEGRAM_INIT'),
        ('TELEGRAM_INIT', 'MAIN_LOOP'),
        ('MAIN_LOOP', 'CURRENT_ADC'),
        ('MAIN_LOOP', 'TEMP_ADC'),
        ('CURRENT_OUTPUT', 'CONTROL_CHECK'),
        ('TEMP_OUTPUT', 'CONTROL_CHECK'),
        ('RELAY_ON', 'SERIAL_OUT'),
        ('RELAY_OFF', 'SERIAL_OUT'),
        ('SEND_ALARM', 'SERIAL_OUT'),
        ('TELEGRAM_POLL', 'HANDLE_CMD'),
        ('HANDLE_CMD', 'CONTROL_CHECK')
    ])

    # 添加循环返回
    dot.edge('SERIAL_OUT', 'MAIN_LOOP',
             label='循环返回',
             style='dashed',
             color='#7D3C98')
    dot.edge('USER_INTERFACE', 'HANDLE_CMD',
             label='用户指令',
             style='dashed',
             color='#7D3C98')

    # 添加注释和说明
    dot.attr(
        label=(
            'ESP32 Temperature Monitoring & Control System\n'
            '=============================================\n\n'
            '系统特点：\n'
            '• 双传感链路并行采样（电流/温度）\n'
            '• 20ms电流周期测量 + 5秒温度采样\n'
            '• 迟滞温度控制（20-25°C范围）\n'
            '• Telegram远程控制与警报\n'
            '• 实时数据可视化与电价计算\n'
            '• 噪声过滤与滑动平均滤波\n\n'
            f'生成时间：{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
        ),
        fontsize='10',
        fontcolor='#2C3E50'
    )

    return dot

def create_data_flow_diagram():
    """创建详细的数据流程图"""

    dot = graphviz.Digraph(
        name='Data_Flow_Diagram',
        format='png',
        graph_attr={
            'rankdir': 'LR',  # 从左到右布局
            'bgcolor': 'transparent',
            'fontname': 'Arial',
            'fontsize': '10',
            'label': 'ESP32数据流与GUI交互图',
            'labelloc': 't',
        },
        node_attr={
            'shape': 'box',
            'style': 'filled',
            'fontname': 'Arial',
            'fontsize': '9',
        }
    )

    # ESP32端节点
    with dot.subgraph(name='cluster_esp32') as esp32:
        esp32.attr(label='ESP32 Microcontroller',
                  style='filled',
                  fillcolor='#E8F6F3',
                  color='#16A085')

        esp32.node('SENSORS', '传感器输入\n- SCT-013电流\n- LM35温度',
                  shape='ellipse',
                  fillcolor='#A3E4D7')

        esp32.node('ADC_PROCESS', 'ADC处理\n- ADS1115 (16位)\n- ESP32 ADC (12位)',
                  shape='box')

        esp32.node('DATA_PROC', '数据处理\n- 电流RMS计算\n- 温度转换\n- 滑动平均滤波',
                  shape='box',
                  fillcolor='#76D7C4')

        esp32.node('CONTROL_LOGIC', '控制逻辑\n- 温度比较\n- 继电器控制\n- 报警检查',
                  shape='diamond',
                  fillcolor='#F7DC6F')

        esp32.node('SERIAL_OUTPUT', '串口格式化\n"Temp: XX.XX °C | Irms: X.XXXXX"',
                  shape='box')

        esp32.node('TELEGRAM_BOT', 'Telegram Bot\n- 消息接收\n- 命令处理\n- 警报发送',
                  shape='box3d',
                  fillcolor='#85C1E9')

    # Python GUI端节点
    with dot.subgraph(name='cluster_gui') as gui:
        gui.attr(label='Python GUI Application',
                style='filled',
                fillcolor='#F4ECF7',
                color='#8E44AD')

        gui.node('SERIAL_READ', '串口数据读取\n- 连接ESP32\n- 解析数据格式',
                shape='box',
                fillcolor='#D2B4DE')

        gui.node('DATA_STORE', '数据存储\n- 温度/电流数组\n- 时间戳记录\n- 会话状态',
                shape='cylinder',
                fillcolor='#BB8FCE')

        gui.node('PRICE_FETCH', '电价API获取\n- REE西班牙API\n- 缓存机制\n- 后备方案',
                shape='parallelogram')

        gui.node('COST_CALC', '成本计算\n- 实时功率计算\n- 累计能耗\n- 费用估算',
                shape='box',
                fillcolor='#A569BD')

        gui.node('VIZ_ENGINE', '可视化引擎\n- Plotly图表\n- 实时更新\n- 交互式界面',
                shape='box3d',
                fillcolor='#8E44AD')

        gui.node('USER_UI', '用户界面\n- Streamlit Web App\n- 控制面板\n- 数据导出',
                shape='component',
                fillcolor='#7D3C98')

    # 数据流连接
    # ESP32内部流
    dot.edges([
        ('SENSORS', 'ADC_PROCESS'),
        ('ADC_PROCESS', 'DATA_PROC'),
        ('DATA_PROC', 'CONTROL_LOGIC'),
        ('CONTROL_LOGIC', 'SERIAL_OUTPUT'),
        ('CONTROL_LOGIC', 'TELEGRAM_BOT'),
    ])

    # ESP32到GUI流
    dot.edge('SERIAL_OUTPUT', 'SERIAL_READ',
             label='115200 baud\n每5秒数据')

    # GUI内部流
    dot.edges([
        ('SERIAL_READ', 'DATA_STORE'),
        ('DATA_STORE', 'PRICE_FETCH'),
        ('PRICE_FETCH', 'COST_CALC'),
        ('COST_CALC', 'VIZ_ENGINE'),
        ('VIZ_ENGINE', 'USER_UI'),
        ('DATA_STORE', 'VIZ_ENGINE'),
    ])

    # 用户交互流
    dot.edge('USER_UI', 'TELEGRAM_BOT',
             label='用户指令\n/命令',
             style='dashed')

    # 添加数据格式说明
    dot.node('DATA_FORMAT',
             '数据格式说明：\n' +
             '• 温度：-40°C to 100°C\n' +
             '• 电流：0.00000 to 15.00000 A\n' +
             '• 采样率：电流20ms，温度5s\n' +
             '• 控制模式：AUTO/MANUAL\n' +
             '• 警报阈值：35°C',
             shape='note',
             fillcolor='#FCF3CF')

    return dot

def save_flowcharts():
    """生成并保存所有流程图"""

    print("生成ESP32系统工作流程图...")
    system_diagram = create_system_flowchart()

    # 渲染流程图
    try:
        system_diagram.render(filename='esp32_system_flowchart',
                              directory='./',
                              cleanup=True,
                              format='png')
        print("✓ 系统流程图已保存为 'esp32_system_flowchart.png'")
    except Exception as e:
        print(f"✗ 生成系统流程图时出错: {e}")
        # 如果graphviz不可用，输出DOT源码
        print("\nDOT源码（可粘贴到在线graphviz工具中）：")
        print("=" * 80)
        print(system_diagram.source)

    print("\n生成数据流程图...")
    data_diagram = create_data_flow_diagram()

    try:
        data_diagram.render(filename='data_flow_diagram',
                            directory='./',
                            cleanup=True,
                            format='png')
        print("✓ 数据流程图已保存为 'data_flow_diagram.png'")
    except Exception as e:
        print(f"✗ 生成数据流程图时出错: {e}")
        print("\nDOT源码（可粘贴到在线graphviz工具中）：")
        print("=" * 80)
        print(data_diagram.source)

    # 创建简单的ASCII流程图
    create_ascii_flowchart()

def create_ascii_flowchart():
    """创建简单的ASCII文本流程图"""

    print("\n" + "=" * 80)
    print("ESP32系统工作流程（ASCII版）")
    print("=" * 80)

    ascii_diagram = """
    ┌─────────────────────────────────────────────────────────────┐
    │                   系统启动与初始化                          │
    ├─────────────────────────────────────────────────────────────┤
    │ 1. ESP32上电                                              │
    │ 2. WiFi连接 (ard-citcea)                                  │
    │ 3. Telegram Bot初始化                                     │
    │ 4. I2C配置 (ADS1115 @ 0x48)                               │
    │ 5. GPIO设置: D5=LED, D7=Relay, A3=Temp                    │
    └─────────────────┬───────────────────────────────────────────┘
                      ↓
    ┌─────────────────────────────────────────────────────────────┐
    │                     主循环开始                              │
    │                   (1ms周期定时器)                          │
    └─────────────────┬───────────────────────────────────────────┘
                      │
           ┌──────────┴──────────┐
           ↓                     ↓
    ┌─────────────┐      ┌─────────────┐
    │电流测量链路 │      │温度测量链路 │
    │(20ms周期)   │      │(5秒周期)    │
    └──────┬──────┘      └──────┬──────┘
           │                     │
    ┌──────▼──────┐      ┌──────▼──────┐
    │ADS1115采样  │      │LM35 ADC采样 │
    │860 SPS      │      │12位分辨率   │
    └──────┬──────┘      └──────┬──────┘
           │                     │
    ┌──────▼──────┐      ┌──────▼──────┐
    │去除DC偏置   │      │电压转换     │
    │V = raw-1.65V│      │(0-3.3V)     │
    └──────┬──────┘      └──────┬──────┘
           │                     │
    ┌──────▼──────┐      ┌──────▼──────┐
    │电流计算     │      │温度计算     │
    │I=V×30A/V    │      │T=V×100°C/V  │
    └──────┬──────┘      └──────┬──────┘
           │                     │
    ┌──────▼──────┐              │
    │RMS累加      │              │
    │sum+=I²×Δt   │              │
    └──────┬──────┘              │
           │                     │
    ┌──────▼──────┐              │
    │每5秒计算RMS │              │
    │Irms=√(50×sum)│              │
    └──────┬──────┘              │
           │                     │
    ┌──────▼──────┐      ┌──────┴──────┐
    │噪声过滤     │      │  数据合并点  │
    │(Irms<0.1→0) │      │ (每5秒同步) │
    └──────┬──────┘      └──────┬──────┘
           │                     │
    ┌──────▼──────┐      ┌──────▼──────┐
    │滑动平均滤波 │      │温度控制逻辑  │
    │250周期平均  │◄─────┤Tmin=20,Tmax=25│
    └──────┬──────┘      └──────┬──────┘
           │                     │
           └─────────┬──────────┘
                     ↓
    ┌─────────────────────────────────────────────────────────────┐
    │                    控制决策                                 │
    ├─────────────────────────────────────────────────────────────┤
    │ IF Temp < 20°C: Relay ON (加热器通电)                      │
    │ IF Temp > 25°C: Relay OFF (加热器断电)                     │
    │ IF Temp ≥ 35°C: 发送Telegram警报                           │
    └─────────────────┬───────────────────────────────────────────┘
                      ↓
    ┌─────────────────────────────────────────────────────────────┐
    │                    数据输出                                 │
    ├─────────────────────────────────────────────────────────────┤
    │ 1. 串口输出: "Temp: XX.XX °C | Irms: X.XXXXX"              │
    │ 2. Telegram响应: /state, /temp, /relay_on, /relay_off      │
    │ 3. GUI更新: 实时图表、数据表格、成本计算                    │
    └─────────────────┬───────────────────────────────────────────┘
                      ↓
    ┌─────────────────────────────────────────────────────────────┐
    │                     循环返回                               │
    └─────────────────────────────────────────────────────────────┘

    系统特性：
    • 双传感链路并行处理
    • 迟滞温度控制防抖动
    • 噪声过滤与滑动平均
    • 多协议通信(串口/WiFi/Telegram)
    • 实时可视化与远程控制
    """

    print(ascii_diagram)

# 安装说明
def installation_instructions():
    """提供安装和运行说明"""

    instructions = """
    ===========================================================================
    安装与运行说明
    ===========================================================================

    1. 安装graphviz（生成流程图需要）:

    Windows:
        • 下载: https://graphviz.org/download/
        • 安装时勾选"Add Graphviz to PATH"

    macOS:
        • brew install graphviz

    Linux:
        • sudo apt-get install graphviz

    2. 安装Python依赖:

        pip install graphviz
        pip install streamlit
        pip install plotly
        pip install pandas
        pip install numpy
        pip install pyserial
        pip install requests
        pip install pytz

    3. 运行流程图生成:

        python flowchart_generator.py

    4. 运行GUI系统:

        streamlit run GUI_final.py

    5. ESP32上传代码:

        • 使用Arduino IDE
        • 选择板子: FireBeetle ESP32
        • 安装库:
          - Universal Telegram Bot Library
          - ArduinoJson
          - Wire
        • 上传sketch_nov_5f_integrated.ino

    ===========================================================================
    系统连接说明
    ===========================================================================

    硬件连接:
        • ADS1115: I2C接口 (SDA=21, SCL=22)
        • 电流传感器: ADS1115 A1端口
        • 温度传感器: ESP32 A3引脚
        • LED指示灯: D5引脚 (绿色)
        • 继电器: D7引脚

    软件连接:
        • WiFi: 配置ssid和password
        • Telegram: 配置BOTtoken和CHAT_ID
        • 串口: ESP32 ↔ Python GUI (COM端口)

    ===========================================================================
    故障排除
    ===========================================================================

    1. 串口连接失败:
        • 检查COM端口设置
        • 检查波特率(115200)
        • 重启ESP32

    2. Telegram无响应:
        • 检查WiFi连接
        • 验证BOTtoken和CHAT_ID
        • 发送/start命令初始化

    3. 传感器数据异常:
        • 检查硬件连接
        • 验证供电电压(3.3V)
        • 检查接地

    4. GUI无数据显示:
        • 检查Python依赖
        • 验证串口通信
        • 查看浏览器控制台
    """

    print(instructions)

if __name__ == "__main__":
    print("=" * 80)
    print("ESP32温度监控系统 - 流程图生成器")
    print("=" * 80)

    # 生成流程图
    save_flowcharts()

    # 显示安装说明
    print("\n" + "=" * 80)
    response = input("是否显示安装说明? (y/n): ")
    if response.lower() == 'y':
        installation_instructions()

    print("\n" + "=" * 80)
    print("流程图生成完成！")
    print("=" * 80)

ESP32温度监控系统 - 流程图生成器
生成ESP32系统工作流程图...
✗ 生成系统流程图时出错: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

DOT源码（可粘贴到在线graphviz工具中）：
digraph ESP32_Temperature_Control_System {
	graph [bgcolor=transparent fontcolor="#2C3E50" fontname=Arial fontsize=12 label="ESP32 Temperature Monitoring & Control System
(2026-01-16)" labeljust=c labelloc=t rankdir=TB]
	node [color="#2C3E50" fillcolor=lightblue fontname=Arial fontsize=10 shape=box style="rounded,filled"]
	edge [color="#34495E" fontname=Arial fontsize=9]
	START [label="系统启动" fillcolor=lightgreen shape=circle style="filled,bold"]
	INIT [label="硬件初始化
- WiFi配置
- I2C初始化
- GPIO设置
- 串口配置" fillcolor="#AED6F1" shape=box3d]
	WIFI_CONNECT [label="WiFi连接
SSID: ard-citcea" fillcolor="#85C1E9" shape=parallelogram]
	TELEGRAM_INIT [label="Telegram Bot初始化
- Token验证
- 安全证书配置" fillcolor="#85C1E9" shape=parallelogram]
	MAIN_LOOP [label="主循环启动
(1ms周期)" fillcolor="#F9E79F" shape=diamond]
	subgraph cluster_current 

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import pytz

def get_ree_price(start_date=None, end_date=None, include_pvpc=False):
    """
    Obtain the electricity price data of Spain from the REE API:
        start_date: Start time (datetime object, default: current time)
        end_date: End time (datetime object, default: current time + 24 hours)
        include_pvpc: Whether to also obtain PVPC electricity price (bool, default: False)

    Return:
    The DataFrame contains the following columns:
        datetime: Time (with time zone)
        spot_price: Spot market price (€/MWh)
        pvpc_price: PVPC electricity price (€/MWh) (if include_pvpc=True)
        spot_price_eur_kwh: Spot market price (€/kWh)
    """

    # API
    endpoint = 'https://apidatos.ree.es'
    get_archives = '/en/datos/mercados/precios-mercados-tiempo-real'

    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'Host': 'apidatos.ree.es'
    }

    if start_date is None:
        spain_tz = pytz.timezone('Europe/Madrid')
        start_date = datetime.now(spain_tz)

    if end_date is None:
        end_date = start_date + timedelta(hours=24)

    start_date_str = start_date.strftime('%Y-%m-%dT%H:%M')
    end_date_str = end_date.strftime('%Y-%m-%dT%H:%M')

    params = {
        'start_date': start_date_str,
        'end_date': end_date_str,
        'time_trunc': 'hour'
    }

    try:
        # send API request
        response = requests.get(
            endpoint + get_archives,
            headers=headers,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data_json = response.json()

        if 'included' not in data_json or len(data_json['included']) < 2:
            raise ValueError("ERROR: Data structure cannot be read")

        spot_market_data = None
        pvpc_data = None

        for item in data_json['included']:
            item_type = item.get('type', '')
            if item_type == 'Precio mercado spot (€/MWh)':
                spot_market_data = item
            elif item_type == 'PVPC (€/MWh)':
                pvpc_data = item

        if spot_market_data is None:
            raise ValueError("ERROR: No market data found")

        spot_values = spot_market_data['attributes']['values']

        data_records = []

        for data_point in spot_values:
            record = {
                'datetime': datetime.fromisoformat(data_point['datetime'].replace('Z', '+00:00')),
                'spot_price': data_point['value'],  # €/MWh
                'spot_price_eur_kwh': data_point['value'] / 1000  # turn to €/kWh
            }

            if include_pvpc and pvpc_data:
                # access for pvpc data if anyone use
                pass

            data_records.append(record)

        df = pd.DataFrame(data_records)
        df = df.sort_values('datetime')
        df = df.reset_index(drop=True)

        return df

    except requests.exceptions.RequestException as e:
        print(f"ERROR: Network connection error: {e}")
        return pd.DataFrame()

    except ValueError as e:
        print(f"ERROR: structure error: {e}")
        return pd.DataFrame()

    except Exception as e:
        print(f"ERROR: Unknown error: {e}")
        return pd.DataFrame()


def get_today_prices():
    """
    get 24h price of today
    """
    spain_tz = pytz.timezone('Europe/Madrid')
    now = datetime.now(spain_tz)
    today_start = now.replace(hour=0, minute=0, second=0, microsecond=0)
    tomorrow_start = today_start + timedelta(days=1)

    if now.hour == 23:
        start_date = now.replace(minute=0, second=0, microsecond=0)
        end_date = start_date + timedelta(hours=24)
    else:
        start_date = now.replace(minute=0, second=0, microsecond=0)
        end_date = start_date + timedelta(hours=24)

    return get_ree_price(start_date=start_date, end_date=end_date)


if __name__ == "__main__":
    print("获取今日电价数据...")

    # 方法1：使用简单函数获取今天24小时数据
    df_today = get_today_prices()

    if not df_today.empty:
        print(f"获取到 {len(df_today)} 小时的电价数据")
        print("\n前5小时数据：")
        print(df_today.head())

        print("\n统计信息：")
        print(f"平均价格: {df_today['spot_price_eur_kwh'].mean():.4f} €/kWh")
        print(f"最高价格: {df_today['spot_price_eur_kwh'].max():.4f} €/kWh")
        print(f"最低价格: {df_today['spot_price_eur_kwh'].min():.4f} €/kWh")
        print(f"当前小时价格: {df_today.iloc[0]['spot_price_eur_kwh']:.4f} €/kWh")

        # 保存到CSV
        filename = f"electricity_prices_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df_today.to_csv(filename, index=False)
        print(f"\n数据已保存到: {filename}")
    else:
        print("未能获取电价数据")

    print("\n" + "="*50 + "\n")

    # 方法2：自定义时间范围
    print("获取自定义时间范围的电价数据...")
    spain_tz = pytz.timezone('Europe/Madrid')

    # 获取明天00:00到后天00:00的数据
    tomorrow = datetime.now(spain_tz) + timedelta(days=1)
    tomorrow_start = tomorrow.replace(hour=0, minute=0, second=0, microsecond=0)
    tomorrow_end = tomorrow_start + timedelta(days=1)

    df_tomorrow = get_ree_price(
        start_date=tomorrow_start,
        end_date=tomorrow_end
    )

    if not df_tomorrow.empty:
        print(f"获取到明天 {len(df_tomorrow)} 小时的电价数据")
        print(f"明天平均价格: {df_tomorrow['spot_price_eur_kwh'].mean():.4f} €/kWh")